# Jour 1 · Chapitre 2 — Scénario : le point du lundi de l'agence, en Python pur
## Formation Beobank · SAS → Python · Corrigé formateur

**L'histoire.** Sophie, conseillère à l'agence de Bruxelles, prépare son point hebdomadaire.
Le poste de l'agence n'a ni SAS ni pandas : seulement Python et le fichier `CTR.csv`.
Elle vous demande :
*"Tu peux me sortir combien de comptes on a, lesquels sont à découvert, et quelques chiffres
clés ? Juste avec Python, sans rien installer."*

**Comment travailler.** Remplacez chaque `______` par le bon mot-clé, la bonne fonction ou la
bonne valeur. Les commentaires indiquent **la notion**, pas la réponse.

📎 **Matière à relire :** `Jour1_Formateur.ipynb` (même dossier) — Modules 2 (variables,
f-strings), 4 (conditions), 5 (boucles), 6 (listes), 8 (dictionnaires), 9 (chaînes, sets),
10 (fonctions, try/except) et 13 (lecture CSV).

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import csv                              # module standard : lecture de fichiers CSV sans pandas
CHEMIN = "../../data/CTR.csv"           # fichier des contrats Beobank
print("Prêt.")

---
## Mission 1 — Un premier calcul pour échauffer

*"Un client a 1219,68 € sur son compte épargne à 3,5 % par an. Ça lui rapporte combien ce mois-ci ?"*

Attention : un solde lu dans un fichier arrive toujours sous forme de **texte**.

In [8]:
# ── Variables, conversion de type, f-string ───────────────────────────────
conseillere = "Sophie"
solde_texte = "1219.68"                 # tel qu'il arrive du fichier : du texte
taux_annuel = 0.035
# Quelle fonction convertit un texte en nombre décimal ?
solde = float(solde_texte)
# Par combien diviser un taux ANNUEL pour obtenir un mois ?
interets_mois = round(solde * taux_annuel / 12, 2)
# Quelle lettre placée devant les guillemets permet d'insérer des {variables} dans le texte ?
print(f"Bonjour {conseillere}, intérêts du mois : {interets_mois:.2f} EUR")   # → 3.56 EUR

Bonjour Sophie, intérêts du mois : 3.56 EUR


---
## Mission 2 — Classer un solde

*"Pour chaque compte, je veux savoir s'il est à découvert, vide, ou créditeur."*

In [9]:
# ── Conditions if / elif / else ───────────────────────────────────────────
def classer_solde(solde):
    # Un compte est à découvert quand son solde est... (quel opérateur de comparaison ?)
    if solde < 0:
        return "Découvert"
    # Quel mot-clé teste une DEUXIÈME condition si la première est fausse ?
    elif solde == 0:
        return "Compte vide"
    # Quel mot-clé couvre tous les autres cas ?
    else:
        return "Créditeur"

print(classer_solde(-8387.77), "|", classer_solde(0.0), "|", classer_solde(361.74))
# → Découvert | Compte vide | Créditeur

Découvert | Compte vide | Créditeur


---
## Mission 3 — Le piège du point

*"Dans le fichier, un solde inconnu est écrit `.` — et ça fait planter mon calcul !"*

`float(".")` lève une erreur. Écrivez une fonction qui renvoie le nombre, ou **rien** si la
conversion échoue.

In [10]:
# ── Gestion d'erreurs try / except ────────────────────────────────────────
def convertir_solde(texte):
    # Quel mot-clé ouvre un bloc qu'on « essaie » d'exécuter ?
    try:
        return float(texte)
    # Quelle erreur Python lève float(".") ? (testez-le dans une cellule si besoin)
    except ValueError:
        # Quelle valeur spéciale Python signifie « rien / absent » ?
        return None

print(convertir_solde("2.82"), convertir_solde("."))      # → 2.82 None

2.82 None


---
## Mission 4 — Lire le fichier des contrats

*"Charge-moi tous les contrats dans une liste, un dictionnaire par contrat."*

In [12]:
# ── Lecture CSV avec le module csv ────────────────────────────────────────
contrats = []
# Quelle fonction de base ouvre un fichier ?
with open(CHEMIN, encoding="utf-8") as fichier:
    # Quel lecteur du module csv renvoie chaque ligne sous forme de dictionnaire ?
    # Quel séparateur utilisent les exports SAS de Beobank ?
    lecteur = csv.DictReader(fichier, delimiter=";")
    for ligne in lecteur:
        # Quelle méthode de liste ajoute un élément à la fin ?
        contrats.append(ligne)

print(len(contrats), "contrats lus")                      # → 200 contrats lus
print("Premier contrat :", contrats[0]["IDT_AC"], contrats[0]["COD_DEV"])

NameError: name 'CHEMIN' is not defined

---
## Mission 5 — Compter les statuts

*"Combien de comptes clôturés, combien de résiliés ?"*

Référentiel : `"4"` = Clôturé, `"6"` = Résilié.

In [6]:
# ── Dictionnaires et boucle de comptage ───────────────────────────────────
statuts = {"4": "Clôturé", "6": "Résilié"}
compteur = {}
for c in contrats:
    # Quelle méthode de dictionnaire lit une clé avec une valeur par défaut si elle n'existe pas ?
    libelle = statuts.get(c["COD_ECV_CTR"], "Inconnu")
    # Au premier passage, le compteur de ce libellé n'existe pas encore : il part de combien ?
    compteur[libelle] = compteur.get(libelle, 0) + 1

print(compteur)                                           # → {'Résilié': 95, 'Clôturé': 105}

{}


---
## Mission 6 — Les découverts

*"Combien de soldes on connaît vraiment, et combien sont dans le rouge ?"*

In [13]:
# ── Listes en compréhension ───────────────────────────────────────────────
soldes = [convertir_solde(c["SLD_CTR"]) for c in contrats]   # None pour les soldes inconnus
# On garde uniquement les soldes connus : comment s'écrit « n'est pas rien » ?
soldes_valides = [s for s in soldes if s is not None]
# Réutilisez la fonction écrite en Mission 2 pour filtrer les découverts
decouverts = [s for s in soldes_valides if classer_solde(s) == "Découvert"]
print(len(soldes_valides), "soldes connus,", len(decouverts), "découverts")   # → 63 soldes connus, 13 découverts
# Quelle fonction renvoie la plus petite valeur d'une liste ?
print("Pire découvert :", min(soldes_valides))

0 soldes connus, 0 découverts


ValueError: min() iterable argument is empty

---
## Mission 7 — Les devises étrangères

*"On a des comptes en devises autres que l'euro ? Lesquelles ?"*

In [14]:
# ── Sets et chaînes de caractères ─────────────────────────────────────────
devises = {c["COD_DEV"] for c in contrats}       # set : chaque devise une seule fois
# Quel opérateur de set retire des éléments d'un autre set ?
etrangeres = sorted(devises - {"EUR"})
# Quelle méthode de chaîne colle les éléments d'une liste avec un séparateur ?
print("Devises étrangères :", ", ".join(etrangeres))   # → AUD, NOK, USD

Devises étrangères : 


---
## 🏁 Mission finale — Le rapport de Sophie

*"Mets-moi tout ça dans un petit encadré que je peux imprimer."*

Format attendu :
```
========================================
  POINT DU LUNDI — AGENCE BRUXELLES
========================================
  Contrats lus      :   200
  Soldes renseignés :    63 (31.5 %)
  En découvert      :    13
  Somme des soldes  :   -743,487.69 EUR
========================================
```

In [15]:
# ── Fonction + f-strings alignées ─────────────────────────────────────────
def pourcentage(partie, total):
    # Une proportion devient un pourcentage en la multipliant par...
    return round(partie / total * 100, 1)

print("=" * 40)
print("  POINT DU LUNDI — AGENCE BRUXELLES")
print("=" * 40)
print(f"  Contrats lus      : {len(contrats):>5}")
print(f"  Soldes renseignés : {len(soldes_valides):>5} ({pourcentage(len(soldes_valides), len(contrats))} %)")
print(f"  En découvert      : {len(decouverts):>5}")
# Quelle fonction additionne tous les éléments d'une liste ?
print(f"  Somme des soldes  : {sum(soldes_valides):>13,.2f} EUR")
print("=" * 40)

  POINT DU LUNDI — AGENCE BRUXELLES
  Contrats lus      :     0


ZeroDivisionError: division by zero

✍️ **Réponse attendue :** la somme ne porte que sur 63 comptes sur 200 (31,5 %) ; quelques gros
découverts suffisent à la rendre négative. Il faut donc présenter ce total avec sa base
(« sur les 63 soldes connus ») et regarder aussi le nombre de découverts, pas seulement la somme.